In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [40]:
# 4 samples, 3 classes
logits = torch.tensor([
    [ 2.0,  0.5, -1.0],   # sample 0
    [-0.2,  1.5,  0.3],   # sample 1
    [ 0.1, -0.4,  2.2],   # sample 2
    [ 1.0,  0.8,  0.2],   # sample 3
])

# true class indices (0, 1, or 2)
targets = torch.tensor([0, 1, 2, 0])
targets = torch.tensor([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 2],
    [1, 0, 0],
])


In [ ]:
gamma = 5
probs = F.softmax(logits, dim=1)
p_t = (probs * targets).sum(dim=1)
# p_t, p_t.shape
torch.log(p_t)
torch.pow((1 - p_t), gamma)

loss = - torch.pow((1 - p_t), gamma) * torch.log(p_t)

tensor([-0.0001, -0.0015, -0.0700, -0.0447])

In [ ]:
class MyFocalLossWithLogits(nn.Module):
    """ 
    My custom implementation of Focal Loss (with logits). Had some help from gpt-5. 
    
    Implements the equation:

        FL(p_t) = −α_t (1 − pₜ)^γ log(p_t)
    
    For the forward method, logits and targets shape is identical. 
    """
    def __init__(self, gamma: float=5.0, reduction: str="mean"):
        super(MyFocalLossWithLogits, self).__init__()
        self.gamma = gamma
        self.reduction = reduction
    
    def get_sample_probabilities(self, logits, targets):
        """ Returns the models estimated probability for the correct class """
        probs = F.softmax(logits, dim=1)
        p_t = (probs * targets).sum(dim=1)
        return p_t
    
    def apply_reduction(self, loss):
        if self.reduction == "mean":
            return loss.mean()
        else:
            return loss.sum()
    
    def forward(self, logits, targets):
        p_t = self.get_sample_probabilities(logits, targets)
        p_t = torch.clamp(p_t, min=1e-7, max=1.0)  
        loss = - ((1 - p_t) ** self.gamma) * torch.log(p_t)
        return self.apply_reduction(loss)

In [ ]:
# Testing BCE
loss = nn.BCEWithLogitsLoss()
pred = torch.randn(3, requires_grad=True)
target = torch.empty(3).random_(2)
# pred, target
output = loss(pred, target)
output
# output.backward()

tensor(0.4915, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)